# Step 3: Data Preprocessing

This notebook handles **Data Preprocessing**:

1. **Preprocessing**: Computes engineered features from raw data
2. **Train/Test Split**: Creates separate tables for model training

## Prerequisites

- Run notebooks 01-02 first

## Imports and Configuration

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import pandas as pd
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

try:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
except:
    print(f"Using current role: {session.get_current_role()}")


DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

## Read Source Data

In [ ]:
print(f"Reading data from {config.full_raw_table}...")
source_df = session.table(config.full_raw_table)
source_df.show(5)

## Create Feature Store

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

fs = FeatureStore(
    session=session, 
    database=DB, 
    name=SCHEMA, 
    default_warehouse=COMPUTE_WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

fs.list_entities().to_pandas()

## Feature Engineering

In [ ]:
import snowflake.snowpark.functions as F

# == Create Shock Index ==
shock_index = F.when(F.col("SYSTOLIC_BP") > 0, F.col("HEART_RATE") / F.col("SYSTOLIC_BP")).otherwise(F.lit(None))

# == Calculate Pulse Pressure ==
pulse_pressure = F.col("SYSTOLIC_BP") - F.col("DIASTOLIC_BP")

# == Calculate BMI Category ==
bmi_category = (F.when(F.col("BMI") < 18.5, F.lit("UNDERWEIGHT"))
    .when(F.col("BMI") < 25, F.lit("NORMAL"))
    .when(F.col("BMI") < 30, F.lit("OVERWEIGHT"))
    .when(F.col("BMI") < 35, F.lit("OBESE"))
    .otherwise(F.lit("SEVERELY_OBESE")))

# == Calculate Vital Sign Severity ==
vital_signs_severity = (
    F.when(F.col("HEART_RATE") > 100, F.lit(1)).otherwise(F.lit(0))
    + F.when(F.col("HEART_RATE") < 60, F.lit(1)).otherwise(F.lit(0))
    + F.when(F.col("RESPIRATORY_RATE") > 20, F.lit(1)).otherwise(F.lit(0))
    + F.when(F.col("OXYGEN_SATURATION") < 94, F.lit(2)).otherwise(F.lit(0))
    + F.when(F.col("TEMPERATURE") > 38.0, F.lit(1)).otherwise(F.lit(0))
    + F.when(F.col("SYSTOLIC_BP") < 90, F.lit(2)).otherwise(F.lit(0))
    + F.when(F.col("SYSTOLIC_BP") > 180, F.lit(1)).otherwise(F.lit(0)))

features_df = source_df.select(
    F.col("PATIENT_ID"),
    F.col("ENCOUNTER_ID"),
    F.col("TIMESTAMP"),
    shock_index.alias("SHOCK_INDEX"),
    pulse_pressure.alias("PULSE_PRESSURE"),
    bmi_category.alias("BMI_CATEGORY"),
    vital_signs_severity.alias("VITAL_SIGNS_SEVERITY"),
)

features_df.show(5)

## Create Feature Store Entity

In [ ]:
patient_entity = Entity(
        name = "PATIENT",
        join_keys = ["PATIENT_ID", "ENCOUNTER_ID"],
        desc = "Computed Patient Indicators"
        )
fs.register_entity(patient_entity)

## Create and Populate Feature View

In [ ]:
# == Create Feature View ==
fv = FeatureView(
    name="PATIENT_FEATURES",
    entities=[patient_entity],
    feature_df=features_df,
    timestamp_col="TIMESTAMP",
    refresh_freq="1 minute",
    desc="Patient features with computed indicators: Shock Index, "
    "Pulse Pressure, BMI Category, and Vital Signs Severity",
)

# == Attach feature description ==
fv.attach_feature_desc(
    {
        "SHOCK_INDEX": "Ratio of heart rate to systolic blood pressure, indicating hemodynamic instability",
        "PULSE_PRESSURE": "Difference between systolic and diastolic blood pressure",
        "BMI_CATEGORY": "Categorical classification of BMI: UNDERWEIGHT, NORMAL, OVERWEIGHT, OBESE, or SEVERELY_OBESE",
        "VITAL_SIGNS_SEVERITY": "Composite severity score based on abnormal vital sign thresholds",
    }
)

# == Register Feature View ==
registered_fv = fs.register_feature_view(
    feature_view=fv,
    version="v1",
    block=True,
    overwrite=True
)


# == View Feature Store Setup ==
print("Entities in Feature Store:")
entities_df = fs.list_entities().to_pandas()
display(entities_df if len(entities_df) > 0 else "No entities found")

print("\nFeature Views in Feature Store:")
fv_df = fs.list_feature_views().to_pandas()
display(fv_df[["NAME", "VERSION", "DESC", "REFRESH_FREQ", "SCHEDULING_STATE"]] if len(fv_df) > 0 else "No feature views found")

## View Feature Set

In [ ]:
features_df = fs.retrieve_feature_values(
    spine_df=source_df, 
    features=[registered_fv], 
    spine_timestamp_col="TIMESTAMP"
    ).to_pandas()

features_df.sample(5)

## Train/Test Split & Write to Tables

In [ ]:
from sklearn.model_selection import train_test_split

training_table = f"{DB}.{SCHEMA}.TRAINING_FEATURES"
test_table = f"{DB}.{SCHEMA}.TEST_FEATURES"
target_col = "RISK_LEVEL"
features_df.columns = [c.upper() for c in features_df.columns]

train_df, test_df = train_test_split(
                    features_df,
                    test_size=0.2,
                    random_state=42,
                    stratify=features_df[target_col],
                )

session.create_dataframe(train_df).write.mode("overwrite").save_as_table(
                training_table
            )

session.create_dataframe(test_df).write.mode("overwrite").save_as_table(
                test_table
            )

session.sql("GRANT OWNERSHIP ON TABLE ML_DEMO_PIPELINE_DB.HEALTHCARE.TRAINING_FEATURES TO ROLE SYSADMIN COPY CURRENT GRANTS;").collect()
session.sql("GRANT OWNERSHIP ON TABLE ML_DEMO_PIPELINE_DB.HEALTHCARE.TEST_FEATURES TO ROLE SYSADMIN COPY CURRENT GRANTS;").collect()

In [ ]:
sample_query = f"""
SELECT PATIENT_ID, ENCOUNTER_ID, 
       HEART_RATE, SYSTOLIC_BP, DIASTOLIC_BP,
       SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY,
       VITAL_SIGNS_SEVERITY, RISK_LEVEL
FROM {training_table}
LIMIT 5
"""

print("Sample of Training Data with Computed Features:")
display(session.sql(sample_query).to_pandas())